# Phase 2 EDA - Houston 2013 (GRSS Data Fusion Contest)

Rubicon's training data source: 144-band CASI hyperspectral cube, LiDAR-derived
height grid, and a 15-class land-cover ground truth over the University of
Houston area (UTM zone 15N, ~2.5 m GSD).

This notebook runs the Phase 2 preprocessing pipeline and sanity-checks it:

1. load + co-register HSI and DEM
2. PCA dimensionality reduction (~99% variance retained)
3. class / severity-proxy distribution
4. sample patch inspection (HSI false-color + DEM overlay)

Run from the `notebooks/` directory with the project venv active.

## Damage-severity proxy (acknowledged simplification)

Houston 2013 has **no damage labels**, so land-cover classes are mapped to
a severity proxy. This is documented in the report as an explicit simplification.

| severity | land-cover classes | rationale |
|---|---|---|
| none | Grass healthy/stressed/synthetic, Tree, Water | vegetated/water, minimal structural exposure |
| moderate | Soil, Residential, Road, Parking Lot 1/2, Tennis Court, Running Track | partial damage, habitable structures |
| severe | Commercial, Highway, Railway | high collapse exposure on hard-built surfaces |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent  # notebooks/ -> ai-engine/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import CLASS_TO_SEVERITY, SEVERITY_NAMES
from src.preprocessing.load_hsi import load_hsi, load_labels
from src.preprocessing.load_lidar import load_lidar_raster
from src.preprocessing.align import align_rasters, assert_aligned
from src.preprocessing.patchify import patch_dataset

# approximate true-colour band indices for the 144-band CASI cube
RGB_BANDS = (63, 37, 22)  # ~670 nm (R), ~550 nm (G), ~480 nm (B)

with open(ROOT / "configs/phase2.yaml") as fh:
    CFG = yaml.safe_load(fh)
DATA = ROOT / CFG["data"]["root"]
print("data root:", DATA)

In [ ]:
hsi, hsi_t, hsi_crs = load_hsi(DATA / CFG["data"]["hsi"])
labels, _, _ = load_labels(DATA / CFG["data"]["labels"])
dem, dem_t, dem_crs = load_lidar_raster(DATA / CFG["data"]["lidar"])

dem_a, hsi_t, hsi_crs = align_rasters(hsi, hsi_t, hsi_crs, dem, dem_t, dem_crs)
assert_aligned(hsi, dem_a)

print("HSI:", hsi.shape, "(rows, cols, bands) - [0, 1] float32")
print("DEM aligned:", dem_a.shape, "| finite cells:", np.isfinite(dem_a).sum())
print("labels:", labels.shape, "| classes present:", np.unique(labels))

In [ ]:
from sklearn.decomposition import PCA

pix = hsi.reshape(-1, hsi.shape[2])
pca_all = PCA().fit(pix)
cum = np.cumsum(pca_all.explained_variance_ratio_)
k99 = int(np.searchsorted(cum, 0.99) + 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, len(cum) + 1), cum)
ax.axhline(0.99, ls="--", c="k", lw=0.8)
ax.axvline(k99, ls=":", c="r", lw=0.8)
ax.set_xlabel("PCA components")
ax.set_ylabel("cumulative explained variance")
print(f"components needed for 99% variance: {k99}")

In [ ]:
from collections import Counter
from src.preprocessing import HOUSTON2013_CLASSES

cnt = Counter(labels.ravel())
cnt.pop(0, None)  # background

sev_counts = {s: 0 for s in SEVERITY_NAMES}
for c, n in cnt.items():
    sev_counts[CLASS_TO_SEVERITY[c]] += n

class_ids = sorted(cnt)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar([HOUSTON2013_CLASSES[c] for c in class_ids], [cnt[c] for c in class_ids])
axes[0].tick_params(axis="x", rotation=45, labelsize=8)
axes[0].set_title("land-cover class pixels")
axes[1].bar(sev_counts.keys(), sev_counts.values(),
            color=["#22C55E", "#F59E0B", "#DC2626"])
axes[1].set_title("severity-proxy pixels")
print("severity pixel counts:", sev_counts)

In [ ]:
patches = patch_dataset(
    hsi, dem_a, labels,
    patch_size=CFG["pipeline"]["patch_size"],
    overlap=CFG["pipeline"]["overlap"],
    min_labeled=CFG["pipeline"]["min_labeled"],
)
print("patches:", patches["hsi"].shape)

rng = np.random.default_rng(0)
picks = rng.choice(len(patches["hsi"]), 6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, i in zip(axes.ravel(), picks):
    rgb = np.stack([patches["hsi"][i, :, :, b] for b in RGB_BANDS], axis=-1)
    ax.imshow(np.clip(rgb, 0, 1))
    ax.imshow(patches["dem"][i], cmap="terrain", alpha=0.45)
    yc = patches["y_class"][i]
    ax.set_title(f"patch {i} | class {yc} -> {CLASS_TO_SEVERITY[yc]}", fontsize=8)
plt.tight_layout()

In [ ]:
comp = np.stack([hsi[:, :, b] for b in RGB_BANDS], axis=-1)
comp = (comp - comp.min()) / max(comp.max() - comp.min(), 1e-9)

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
axes[0].imshow(comp)
axes[0].set_title("HSI false-colour")
axes[1].imshow(dem_a, cmap="terrain")
axes[1].set_title("LiDAR DEM")
axes[2].imshow(labels, cmap="tab20", vmin=0, vmax=15)
axes[2].set_title("land-cover labels")
plt.tight_layout()

## Notes / next steps

- If PCA needs only a handful of components, the fused model can be small and fast.
- The DEM/HSI alignment should be visually verified in the two left panels above.
- Production flow: `python -m src.preprocessing.dataset --config configs/phase2.yaml`
  writes the aligned, PCA-reduced, 70/15/15-stratified tensors to `data/processed/`.
- Phase 3 consumes `X_hsi` (channels-first) + `X_dem` as a two-branch input.